# Notebook Overview — Generate CLIP Video Representations

## Purpose

Generate reusable CLIP-based video representations for NExT-QA videos. These video representations provide a pretrained visual embedding baseline for downstream representation-based VideoQA experiments and comparison against self-supervised autoencoder video representations.

## Inputs

* Shared project configuration and constants.
* NExT-QA annotation files used to identify referenced videos.
* NExT-QA video files.
* CLIP video dataset mode configuration controlling development or full-dataset generation.
* Pretrained CLIP image encoder model from the Hugging Face Transformers library.

## Outputs

* CLIP video representation dataset containing one 512-dimensional embedding per selected video.
* CLIP video representation summary report.
* Validation results confirming representation dataset integrity.
* Sample representation records for qualitative verification.

## Processing Workflow

1. Initialize the notebook environment and load shared project configuration.
2. Configure shared CLIP video representation generation.
3. Verify the runtime environment and required software dependencies.
4. Prepare the selected NExT-QA video input dataset.
5. Load the pretrained CLIP image encoder model.
6. Generate normalized CLIP video representations from sampled video frames.
7. Validate the generated representation dataset.
8. Save the representation dataset locally and copy the full shared artifact to Google Drive when selected.
9. Generate a summary report describing the completed representation dataset.
10. Display representative video representation records for verification.
11. Summarize notebook outputs and generated artifacts.

## Notes

This notebook generates pretrained CLIP video representations only. Text representations are produced separately by Notebook 05, and autoencoder video representations are produced separately by Notebook 04.

CLIP video representations are generated from a fixed pretrained CLIP image encoder and are not experiment-dependent. Therefore, these artifacts are stored once in the shared representations directory rather than under an experiment-specific directory.

Development mode generates a small sampled subset of videos for validation. Full mode generates the persistent shared CLIP video embedding dataset for all NExT-QA videos referenced by the annotation files.


### 🔷 Step 1 — Initialize Environment for CLIP Video Representation Generation

* Mount Google Drive and prepare the Colab execution environment.
* Clone the private project repository and move into the repository directory.
* Load shared project configuration constants and utility modules.
* Verify required project paths and local output directories.
* Load NExT-QA annotation metadata used for CLIP text representation generation.
* Combine annotation split files into a unified dataset for downstream processing.
* Display dataset split summary information when verbose output is enabled.
* Prepare the notebook environment for CLIP-based text representation generation.

In [ ]:
# ============================================================
# Step 1: Initialize Environment for CLIP Video Representation Generation
# ============================================================

# ============================================================
# Notebook Execution Controls
# ============================================================

# False -> Generate embeddings for the development subset.
# True  -> Generate embeddings for the complete NExT-QA dataset.
GENERATE_FULL_DATASET = True

VERBOSE = True
REQUIRE_L4_GPU = True
EXPECTED_NEXTQA_VIDEO_COUNT = 5440

# ------------------------------------------------------------
# IMPORTS (must happen before path usage)
# ------------------------------------------------------------

import os
import shutil
import time
from pathlib import Path

import pandas as pd

from google.colab import userdata, drive

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# CLONE REPOSITORY
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# CONFIG LOAD
# ------------------------------------------------------------

print("\nLoading project configuration...")

from src.videoqa_representation_config import *

# ------------------------------------------------------------
# LOAD MODULES
# ------------------------------------------------------------

from src.nextqa_video_cache import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# REQUIRED PATH CHECK
# ------------------------------------------------------------

required_paths = [
    Path("src"),
    Path("datasets"),
    QUESTIONS_DIR,
    METADATA_DIR,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# RESTORE VIDEO CACHE
# ------------------------------------------------------------

print("\nChecking local NExT-QA video cache...")

existing_video_files = sorted(VIDEOS_DIR.rglob("*.mp4"))

if len(existing_video_files) == EXPECTED_NEXTQA_VIDEO_COUNT:

    print("Local video cache already available.")
    print(f"Videos found: {len(existing_video_files):,}")

else:

    print("Video cache missing — restoring from Google Drive...")

    DRIVE_DATASET_DIR = GOOGLE_DRIVE_ROOT / "NExT-QA"
    DRIVE_RELEASES_DIR = DRIVE_DATASET_DIR / "releases"

    LOCAL_ARCHIVE_DIR = DATASET_DIR / "archives"
    LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    COMBINED_ARCHIVE_NAME = "NExTVideo_combined.zip"

    DRIVE_COMBINED_ARCHIVE_PATH = DRIVE_RELEASES_DIR / COMBINED_ARCHIVE_NAME
    LOCAL_ARCHIVE_PATH = LOCAL_ARCHIVE_DIR / COMBINED_ARCHIVE_NAME

    if not DRIVE_COMBINED_ARCHIVE_PATH.exists():
        raise FileNotFoundError(
            f"Missing dataset archive in Drive: {DRIVE_COMBINED_ARCHIVE_PATH}"
        )

    print("Copying dataset archive from Drive...")

    shutil.copy2(DRIVE_COMBINED_ARCHIVE_PATH, LOCAL_ARCHIVE_PATH)

    print("Extracting video archive...")

    extract_nextqa_video_archive(
        combined_archive_path=LOCAL_ARCHIVE_PATH,
        local_videos_dir=VIDEOS_DIR,
        force_extract=False,
        verbose=VERBOSE,
    )

    print("Video cache restored.")

# ------------------------------------------------------------
# LOAD METADATA
# ------------------------------------------------------------

print("\nLoading NExT-QA metadata and video inventory...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

video_inventory_df = build_nextqa_video_inventory(
    videos_dir=VIDEOS_DIR,
    verbose=VERBOSE,
)

annotations_with_videos_df = attach_video_inventory_to_annotations(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

coverage_summary = verify_annotation_video_coverage(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

print("\nDataset metadata ready.")
print(f"Annotation records: {len(annotations_df):,}")
print(f"Video inventory   : {len(video_inventory_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook is ready for CLIP video representation generation.")



### 🔷 Step 2 — Define CLIP Video Representation Configuration

* Load shared constants from the central project configuration file.
* Configure CLIP-based video representation generation for either development or full-dataset execution.
* Specify whether to generate representations for a development video subset or all referenced NExT-QA videos.
* Define the NExT-QA evaluation split, development subset size, and random seed used during development execution.
* Configure the CLIP image encoder model used for video frame representation generation.
* Define the frame sampling strategy used to convert videos into pooled CLIP embeddings.
* Configure the shared output location for reusable CLIP video representation artifacts.
* Display the active CLIP video representation configuration prior to model loading.


In [ ]:
# ============================================================
# Step 2: Define CLIP Video Representation Configuration
# ============================================================

print("Defining CLIP video representation configuration...")

# ------------------------------------------------------------
# Validate required configuration values
# ------------------------------------------------------------

required_config_values = [
    "CLIP_VIDEO_MODEL_NAME",
    "CLIP_VIDEO_REPRESENTATION_SCOPE",
    "CLIP_VIDEO_REPRESENTATION_SOURCE",
    "CLIP_VIDEO_FRAMES_PER_VIDEO",
    "CLIP_VIDEO_FRAME_BATCH_SIZE",
    "CLIP_VIDEO_POOLING_METHOD",
    "CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV",
    "CLIP_VIDEO_SUMMARY_LOCAL_CSV",
    "CLIP_VIDEO_REPRESENTATIONS_DRIVE_DIR",
    "CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV",
    "CLIP_VIDEO_SUMMARY_DRIVE_CSV",
    "VIDEOS_DIR",
    "EVALUATION_SPLIT",
    "DEVELOPMENT_SUBSET_SIZE",
    "RANDOM_SEED",
    "GENERATE_FULL_DATASET",
]

missing_config_values = [
    name for name in required_config_values
    if name not in globals()
]

if missing_config_values:
    raise NameError(
        "Missing required CLIP video configuration values: "
        + ", ".join(missing_config_values)
    )

# ------------------------------------------------------------
# Dataset generation configuration
# ------------------------------------------------------------

evaluation_split = EVALUATION_SPLIT
development_video_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

# ------------------------------------------------------------
# Video directory validation
# ------------------------------------------------------------

NEXTQA_VIDEO_SOURCE_DIR = VIDEOS_DIR

if not NEXTQA_VIDEO_SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"NExT-QA video directory not found: {NEXTQA_VIDEO_SOURCE_DIR}"
    )

video_files = sorted(NEXTQA_VIDEO_SOURCE_DIR.rglob("*.mp4"))

if not video_files:
    raise FileNotFoundError(
        f"No MP4 video files found under: {NEXTQA_VIDEO_SOURCE_DIR}"
    )

# ------------------------------------------------------------
# Output directory setup
# ------------------------------------------------------------

CLIP_VIDEO_LOCAL_OUTPUT_DIR = (
    CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV.parent
)

CLIP_VIDEO_LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Drive directory is created only during full-dataset artifact copy.

# ------------------------------------------------------------
# Display active configuration
# ------------------------------------------------------------

clip_video_config_summary = {
    "artifact_scope": "shared",
    "clip_video_model": CLIP_VIDEO_MODEL_NAME,
    "representation_scope": CLIP_VIDEO_REPRESENTATION_SCOPE,
    "representation_source": CLIP_VIDEO_REPRESENTATION_SOURCE,
    "generate_full_dataset": GENERATE_FULL_DATASET,
    "evaluation_split": evaluation_split,
    "development_video_subset_size": development_video_subset_size,
    "random_seed": random_seed,
    "frames_per_video": CLIP_VIDEO_FRAMES_PER_VIDEO,
    "frame_batch_size": CLIP_VIDEO_FRAME_BATCH_SIZE,
    "pooling_method": CLIP_VIDEO_POOLING_METHOD,
    "video_source_directory": str(NEXTQA_VIDEO_SOURCE_DIR),
    "video_files_found": len(video_files),
    "local_output_directory": str(CLIP_VIDEO_LOCAL_OUTPUT_DIR),
    "drive_output_directory": str(CLIP_VIDEO_REPRESENTATIONS_DRIVE_DIR),
}

clip_video_config_df = pd.DataFrame(
    clip_video_config_summary.items(),
    columns=["Configuration Item", "Value"],
)

print("CLIP video representation configuration defined.")
display(clip_video_config_df)



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Verify the active Python runtime, operating system, and PyTorch installation.
* Confirm that CUDA is available and validate the required NVIDIA L4 GPU when enabled.
* Display detected GPU hardware and available GPU memory.
* Verify that the Hugging Face Transformers library is installed and available.
* Confirm that OpenCV is installed for video frame extraction.
* Confirm that the required CLIP model and processor classes can be imported successfully.
* Validate that the runtime environment satisfies all software and hardware requirements.
* Display a runtime verification summary before loading the CLIP video model.


In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import platform
import sys

import torch

print("Verifying runtime environment...")
print("-" * 60)

print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")
print(f"PyTorch Version: {torch.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available : {cuda_available}")

if REQUIRE_L4_GPU:

    if not cuda_available:
        raise RuntimeError(
            "CUDA GPU is required for CLIP video representation generation."
        )

    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU            : {gpu_name}")

    if "L4" not in gpu_name:
        raise RuntimeError(
            f"NVIDIA L4 GPU required. Detected: {gpu_name}"
        )

    gpu_properties = torch.cuda.get_device_properties(0)

    gpu_memory_gb = (
        gpu_properties.total_memory
        / (1024 ** 3)
    )

    print(f"GPU Memory     : {gpu_memory_gb:.1f} GB")

try:
    import transformers
    print(f"Transformers   : {transformers.__version__}")

except ImportError:
    raise ImportError("The transformers package is required.")

try:
    import cv2
    print(f"OpenCV         : {cv2.__version__}")

except ImportError:
    raise ImportError(
        "The cv2 package is required for video frame extraction."
    )

try:
    from transformers import (
        CLIPModel,
        CLIPProcessor,
    )

    print("CLIP classes   : Available")

except ImportError:
    raise ImportError(
        "Unable to import CLIPModel and CLIPProcessor."
    )

print("\nRuntime verification complete.")
print("-" * 60)
print("Environment is ready for CLIP video representation generation.")


### 🔷 Step 4 — Prepare CLIP Video Input Dataset

* Select either the configured development video subset or all referenced NExT-QA videos.
* Validate that annotation metadata contains the required split and video identifier fields.
* Generate a reproducible development video subset when development execution mode is selected.
* Build one normalized input record per unique selected video.
* Resolve each selected video identifier to an available video file.
* Validate that all selected video files exist before CLIP encoding begins.
* Attach representation metadata required for downstream CLIP video embedding generation.
* Display summary statistics and representative video input records for verification.


In [ ]:
# ============================================================
# Step 4: Prepare CLIP Video Input Dataset
# ============================================================

import pandas as pd
from pathlib import Path

print("Preparing CLIP video input dataset...")

required_annotation_columns = [
    "split",
    VIDEO_ID_COLUMN,
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        f"annotations_df is missing required columns: {missing_annotation_columns}"
    )

if GENERATE_FULL_DATASET:

    selected_annotations_df = (
        annotations_df
        .copy()
        .reset_index(drop=True)
    )

else:

    selected_annotations_df = annotations_df[
        annotations_df["split"] == evaluation_split
    ].copy()

    if len(selected_annotations_df) == 0:
        raise ValueError(f"No records found for split: {evaluation_split}")

if len(selected_annotations_df) == 0:
    raise RuntimeError("No annotation records were selected.")

selected_annotations_df[VIDEO_ID_COLUMN] = (
    selected_annotations_df[VIDEO_ID_COLUMN]
    .astype(str)
)

video_split_df = (
    selected_annotations_df[
        ["split", VIDEO_ID_COLUMN]
    ]
    .drop_duplicates()
    .rename(columns={VIDEO_ID_COLUMN: "video"})
)

video_summary_df = (
    video_split_df
    .groupby("video")["split"]
    .apply(lambda values: ",".join(sorted(set(values.astype(str)))))
    .reset_index(name="split")
)

if not GENERATE_FULL_DATASET:

    sample_size = min(
        development_video_subset_size,
        len(video_summary_df),
    )

    video_summary_df = (
        video_summary_df
        .sample(
            n=sample_size,
            random_state=random_seed,
        )
        .sort_values("video")
        .reset_index(drop=True)
    )

else:

    video_summary_df = (
        video_summary_df
        .sort_values("video")
        .reset_index(drop=True)
    )

if video_summary_df.empty:
    raise RuntimeError("No CLIP video input records were generated.")

print(f"Indexing video files under: {NEXTQA_VIDEO_SOURCE_DIR}")

video_file_index = {}

for extension in CLIP_VIDEO_FILE_EXTENSIONS:
    for video_path in NEXTQA_VIDEO_SOURCE_DIR.rglob(f"*{extension}"):
        video_file_index[video_path.stem] = video_path

if len(video_file_index) == 0:
    raise FileNotFoundError(
        f"No video files found under {NEXTQA_VIDEO_SOURCE_DIR}"
    )

video_records = []

for row in video_summary_df.to_dict("records"):

    video_id = str(row["video"])
    video_path = video_file_index.get(video_id)

    video_records.append(
        {
            "record_id": f"{video_id}_clip_video",
            "video": video_id,
            "split": row["split"],
            "video_path": str(video_path) if video_path is not None else None,
            "representation_source": CLIP_VIDEO_REPRESENTATION_SOURCE,
            "frame_sampling_method": "uniform",
            "frames_requested": CLIP_VIDEO_FRAMES_PER_VIDEO,
            "pooling_method": CLIP_VIDEO_POOLING_METHOD,
        }
    )

video_input_df = pd.DataFrame(video_records)

required_video_columns = [
    "record_id",
    "video",
    "split",
    "video_path",
    "representation_source",
    "frame_sampling_method",
    "frames_requested",
    "pooling_method",
]

missing_video_columns = [
    col for col in required_video_columns
    if col not in video_input_df.columns
]

if missing_video_columns:
    raise ValueError(
        f"video_input_df is missing required columns: {missing_video_columns}"
    )

missing_video_path_count = (
    video_input_df["video_path"]
    .isna()
    .sum()
)

if missing_video_path_count > 0:

    missing_video_ids = (
        video_input_df
        .loc[
            video_input_df["video_path"].isna(),
            "video"
        ]
        .head(20)
        .tolist()
    )

    raise FileNotFoundError(
        f"Missing video files for {missing_video_path_count} selected videos. "
        f"First missing video IDs: {missing_video_ids}"
    )

duplicate_record_id_count = (
    video_input_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_record_id_count > 0:
    raise ValueError(
        f"Found {duplicate_record_id_count} duplicate record_id values."
    )

input_splits = sorted(
    selected_annotations_df["split"]
    .astype(str)
    .unique()
    .tolist()
)

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

print("CLIP video input dataset prepared successfully.")

print(f"Dataset mode          : {dataset_mode_label}")
print(f"Source split          : {evaluation_split}")
print(f"Selected annotations  : {len(selected_annotations_df):,}")

referenced_video_count = (
    selected_annotations_df["video"]
    .nunique()
)

print(f"Referenced videos     : {referenced_video_count:,}")
print(f"Video input records   : {len(video_input_df):,}")
print(f"Available videos      : {len(video_inventory_df):,}")
print(f"Frames/video          : {CLIP_VIDEO_FRAMES_PER_VIDEO}")
print(f"Video source dir      : {NEXTQA_VIDEO_SOURCE_DIR}")

print("\nVideo Input Preview:")
display(video_input_df.head(10))



### 🔷 Step 5 — Load CLIP Video Model

* Select the appropriate computation device for CLIP video representation generation.
* Load the pretrained CLIP image encoder model from the Hugging Face Transformers library.
* Load the corresponding CLIP processor used for image preprocessing.
* Configure the CLIP model for inference by switching to evaluation mode.
* Verify the projected image embedding dimension.
* Perform a sample inference to confirm successful image embedding generation.
* Display model configuration details and verification results prior to batch representation generation.


In [ ]:
# ============================================================
# Step 5: Load CLIP Video Model
# ============================================================

import torch
from PIL import Image
import numpy as np

from transformers import (
    CLIPModel,
    CLIPProcessor,
)

print("Loading CLIP video model...")

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Device: {device}")

clip_model = CLIPModel.from_pretrained(
    CLIP_VIDEO_MODEL_NAME
)

clip_model.to(device)
clip_model.eval()

clip_processor = CLIPProcessor.from_pretrained(
    CLIP_VIDEO_MODEL_NAME
)

video_embedding_dimension = (
    clip_model.config.projection_dim
)

vision_hidden_size = (
    clip_model.vision_model.config.hidden_size
)

print("\nCLIP video model loaded successfully.")
print(f"Model                     : {CLIP_VIDEO_MODEL_NAME}")
print(f"Embedding dimension       : {video_embedding_dimension}")
print(f"Vision hidden size        : {vision_hidden_size}")
print(f"Model device              : {device}")

sample_image = Image.fromarray(
    np.zeros(
        (224, 224, 3),
        dtype=np.uint8,
    )
)

sample_inputs = clip_processor(
    images=[sample_image],
    return_tensors="pt",
)

sample_inputs = {
    key: value.to(device)
    for key, value in sample_inputs.items()
}

with torch.no_grad():

    sample_outputs = clip_model.vision_model(
        pixel_values=sample_inputs["pixel_values"]
    )

    sample_pooled_output = sample_outputs.pooler_output

    sample_features = clip_model.visual_projection(
        sample_pooled_output
    )

if sample_features.ndim != 2:
    raise ValueError(
        "CLIP image feature verification failed. "
        f"Expected 2D tensor, found shape {tuple(sample_features.shape)}."
    )

if sample_features.shape[0] != 1:
    raise ValueError(
        "CLIP image feature verification failed. "
        f"Expected batch size 1, found {sample_features.shape[0]}."
    )

if sample_features.shape[1] != video_embedding_dimension:
    raise ValueError(
        "CLIP image feature verification failed. "
        f"Expected embedding dimension {video_embedding_dimension}, "
        f"found {sample_features.shape[1]}."
    )

print(
    f"Verification embedding shape : "
    f"{tuple(sample_features.shape)}"
)

print("\nCLIP video model is ready for representation generation.")



### 🔷 Step 6 — Generate CLIP Video Representations

* Sample a fixed number of frames uniformly from each selected video.
* Encode sampled frames using the pretrained CLIP image encoder.
* Normalize each frame embedding vector.
* Aggregate frame embeddings into one pooled video embedding per video.
* Associate each embedding with its corresponding video record and representation metadata.
* Construct the complete CLIP video representation dataset for downstream VideoQA experiments.
* Verify that the expected embedding dimensions were generated successfully.
* Display summary statistics describing the completed video representation generation process.


In [ ]:
# ============================================================
# Step 6: Generate CLIP Video Representations
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

print("Generating CLIP video representations...")

if "video_input_df" not in globals():
    raise NameError("video_input_df was not found. Run Step 4 first.")

if "clip_model" not in globals():
    raise NameError("clip_model was not found. Run Step 5 first.")

if "clip_processor" not in globals():
    raise NameError("clip_processor was not found. Run Step 5 first.")

def sample_video_frames(
    video_path,
    frames_per_video,
):

    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        raise RuntimeError(f"Unable to open video file: {video_path}")

    total_frames = int(
        capture.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        capture.get(cv2.CAP_PROP_FPS)
    )

    if total_frames <= 0:
        capture.release()
        raise RuntimeError(f"Video contains no readable frames: {video_path}")

    frame_indices = np.linspace(
        0,
        total_frames - 1,
        num=min(frames_per_video, total_frames),
        dtype=int,
    )

    sampled_frames = []

    for frame_index in frame_indices:

        capture.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_index),
        )

        success, frame_bgr = capture.read()

        if not success or frame_bgr is None:
            continue

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        sampled_frames.append(
            Image.fromarray(frame_rgb)
        )

    capture.release()

    if len(sampled_frames) == 0:
        raise RuntimeError(f"No frames could be sampled from: {video_path}")

    duration_seconds = (
        total_frames / fps
        if fps and fps > 0
        else None
    )

    return sampled_frames, total_frames, fps, duration_seconds


def encode_frames_with_clip(
    frames,
):

    frame_embeddings = []

    for start_idx in range(
        0,
        len(frames),
        CLIP_VIDEO_FRAME_BATCH_SIZE,
    ):

        batch_frames = frames[
            start_idx:start_idx + CLIP_VIDEO_FRAME_BATCH_SIZE
        ]

        inputs = clip_processor(
            images=batch_frames,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            image_outputs = clip_model.vision_model(
                pixel_values=inputs["pixel_values"]
            )

            pooled_output = image_outputs.pooler_output

            image_features = clip_model.visual_projection(
                pooled_output
            )

            if not isinstance(image_features, torch.Tensor):
                raise TypeError(
                    "CLIP frame encoding failed. "
                    f"Expected torch.Tensor, found {type(image_features)}."
                )

            if image_features.ndim != 2:
                raise ValueError(
                    "CLIP frame encoding failed. "
                    f"Expected 2D tensor, found shape {tuple(image_features.shape)}."
                )

            if image_features.shape[1] != video_embedding_dimension:
                raise ValueError(
                    "CLIP frame encoding failed. "
                    f"Expected embedding dimension {video_embedding_dimension}, "
                    f"found {image_features.shape[1]}."
                )

            image_features = image_features / image_features.norm(
                dim=-1,
                keepdim=True,
            )

        frame_embeddings.append(
            image_features
            .detach()
            .cpu()
            .numpy()
        )

    return np.vstack(frame_embeddings)


def pool_frame_embeddings(
    frame_embeddings,
):

    if CLIP_VIDEO_POOLING_METHOD != "mean":
        raise ValueError(
            f"Unsupported pooling method: {CLIP_VIDEO_POOLING_METHOD}"
        )

    video_embedding = frame_embeddings.mean(axis=0)

    embedding_norm = np.linalg.norm(video_embedding)

    if embedding_norm == 0:
        raise RuntimeError("Generated zero-norm video embedding.")

    video_embedding = video_embedding / embedding_norm

    return video_embedding


clip_model.eval()

records = []
failed_records = []
start_time = time.time()

for row in tqdm(
    video_input_df.to_dict("records"),
    desc="Encoding CLIP videos",
):

    try:

        video_path = Path(row["video_path"])

        sampled_frames, total_frames, fps, duration_seconds = sample_video_frames(
            video_path=video_path,
            frames_per_video=CLIP_VIDEO_FRAMES_PER_VIDEO,
        )

        frame_embeddings = encode_frames_with_clip(
            frames=sampled_frames,
        )

        video_embedding = pool_frame_embeddings(
            frame_embeddings=frame_embeddings,
        )

        record = dict(row)

        record["frames_sampled"] = len(sampled_frames)
        record["video_frame_count"] = total_frames
        record["video_fps"] = fps
        record["video_duration_seconds"] = duration_seconds
        record["clip_video_model"] = CLIP_VIDEO_MODEL_NAME
        record["embedding_dimension"] = len(video_embedding)

        for i, value in enumerate(video_embedding):
            record[f"clip_video_{i:03d}"] = float(value)

        records.append(record)

    except Exception as error:

        failed_records.append(
            {
                "video": row.get("video"),
                "video_path": row.get("video_path"),
                "error": str(error),
            }
        )

        if VERBOSE:
            print(
                f"Failed video {row.get('video')}: {error}"
            )

elapsed_time = time.time() - start_time

if failed_records:
    failed_video_df = pd.DataFrame(failed_records)

    display(failed_video_df.head(20))

    raise RuntimeError(
        f"CLIP video representation generation failed for "
        f"{len(failed_records)} videos."
    )

clip_video_representation_df = pd.DataFrame(records)

if clip_video_representation_df.empty:
    raise RuntimeError(
        "No CLIP video representations were generated."
    )

clip_video_columns = [
    col
    for col in clip_video_representation_df.columns
    if col.startswith("clip_video_")
    and col.replace("clip_video_", "").isdigit()
]

if len(clip_video_columns) != video_embedding_dimension:
    raise ValueError(
        f"Expected {video_embedding_dimension} CLIP video embedding columns, "
        f"found {len(clip_video_columns)}."
    )

if len(clip_video_columns) == 0:
    raise ValueError(
        "No CLIP video embedding columns were generated."
    )

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

print("\nCLIP video representation generation complete.")
print(f"Dataset mode         : {dataset_mode_label}")
print(f"Input video records  : {len(video_input_df):,}")
print(f"Embedding records    : {len(clip_video_representation_df):,}")
print(f"Embedding dimensions : {len(clip_video_columns):,}")
print(f"Frames/video target  : {CLIP_VIDEO_FRAMES_PER_VIDEO}")
print(f"Frame batch size     : {CLIP_VIDEO_FRAME_BATCH_SIZE}")
print(f"Elapsed time         : {elapsed_time:.1f} seconds")
print(
    f"Average/video        : "
    f"{elapsed_time / len(clip_video_representation_df):.3f} seconds"
)


### 🔷 Step 7 — Validate CLIP Video Representation Dataset

* Verify that the CLIP video representation dataset was generated successfully.
* Validate the number of video representation records and unique videos.
* Confirm that the expected CLIP video embedding dimensions were produced.
* Verify that all embedding values are present and numeric.
* Check for duplicate representation records within the generated dataset.
* Confirm that sampled frame counts are positive for all videos.
* Verify that the representation dataset is ready for downstream VideoQA experiments.


In [ ]:
# ============================================================
# Step 7: Validate CLIP Video Representation Dataset
# ============================================================

import pandas as pd

print("Validating CLIP video representation dataset...")

if "clip_video_representation_df" not in globals():
    raise NameError(
        "clip_video_representation_df was not found. Run Step 6 first."
    )

if "clip_video_columns" not in globals():
    raise NameError(
        "clip_video_columns was not found. Run Step 6 first."
    )

validation_summary = {
    "representation_records": len(clip_video_representation_df),
    "unique_videos": clip_video_representation_df["video"].nunique(),
    "embedding_dimensions": len(clip_video_columns),
    "missing_embedding_values": (
        clip_video_representation_df[clip_video_columns]
        .isna()
        .sum()
        .sum()
    ),
    "non_numeric_embedding_columns": sum(
        not pd.api.types.is_numeric_dtype(clip_video_representation_df[col])
        for col in clip_video_columns
    ),
    "duplicate_record_ids": clip_video_representation_df["record_id"].duplicated().sum(),
    "duplicate_videos": clip_video_representation_df["video"].duplicated().sum(),
    "missing_video_paths": clip_video_representation_df["video_path"].isna().sum(),
    "non_positive_sampled_frame_counts": (
        clip_video_representation_df["frames_sampled"]
        .le(0)
        .sum()
    ),
}

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Value"],
)

display(validation_df)

if validation_summary["missing_embedding_values"] > 0:
    raise ValueError("Missing values detected in CLIP video representations.")

if validation_summary["non_numeric_embedding_columns"] > 0:
    raise ValueError("Non-numeric embedding columns detected.")

if validation_summary["duplicate_record_ids"] > 0:
    raise ValueError("Duplicate record IDs detected.")

if validation_summary["duplicate_videos"] > 0:
    raise ValueError("Duplicate video representation records detected.")

if validation_summary["missing_video_paths"] > 0:
    raise ValueError("Missing video paths detected.")

if validation_summary["non_positive_sampled_frame_counts"] > 0:
    raise ValueError("Videos with non-positive sampled frame counts detected.")

if validation_summary["representation_records"] != validation_summary["unique_videos"]:
    raise ValueError(
        "Expected exactly one CLIP video representation per unique video."
    )

print(
    "\nRepresentation validation passed. "
    "No missing, non-numeric, duplicate, or invalid video records detected."
)


### 🔷 Step 8 — Save CLIP Video Representation Files

* Create the local CLIP video representation output directory when necessary.
* Save the generated CLIP video representation dataset to local project storage.
* Verify that the local representation dataset was written successfully.
* Copy the representation dataset to the shared Google Drive output directory when full-dataset generation is selected.
* Confirm the number of generated video representation records and embedding dimensions.
* Display the local and shared output locations for downstream representation-based VideoQA experiments.


In [ ]:
# ============================================================
# Step 8: Save CLIP Video Representation Files
# ============================================================

import shutil

print("Saving CLIP video representation files...")

if "clip_video_representation_df" not in globals():
    raise NameError(
        "clip_video_representation_df was not found. Run Step 6 first."
    )

if "clip_video_columns" not in globals():
    raise NameError(
        "clip_video_columns was not found. Run Step 6 first."
    )

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

CLIP_VIDEO_LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

clip_video_representation_df.to_csv(
    CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV,
    index=False,
)

if not CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create local file: "
        f"{CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV}"
    )

drive_artifact_written = False

if GENERATE_FULL_DATASET:

    CLIP_VIDEO_REPRESENTATIONS_DRIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV,
        CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV,
    )

    if not CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV.exists():
        raise FileNotFoundError(
            f"Failed to create Drive file: "
            f"{CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV}"
        )

    drive_artifact_written = True

else:

    print(
        "Development mode complete. Local representation file was "
        "created, but the persistent shared Drive artifact was not "
        "overwritten."
    )

print("\nCLIP video representation dataset saved successfully.")
print(f"Dataset mode         : {dataset_mode_label}")
print(f"Local output file    : {CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV}")

if drive_artifact_written:
    print(f"Drive output file    : {CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV}")
else:
    print("Drive output file    : not written in development mode")

print(f"Representation rows  : {len(clip_video_representation_df):,}")
print(f"Embedding dimensions : {len(clip_video_columns):,}")
print(f"Unique videos        : {clip_video_representation_df['video'].nunique():,}")


### 🔷 Step 9 — Generate CLIP Video Representation Summary Report

* Generate summary statistics describing the completed CLIP video representation dataset.
* Summarize the number of generated video representations.
* Report the number of unique videos and input dataset splits included in the generated representation dataset.
* Verify the generated CLIP video embedding dimensionality and check for missing embedding values.
* Record shared representation metadata required for downstream representation-based VideoQA experiments.
* Save the representation summary report to local project storage.
* Copy the summary report to the shared Google Drive output directory when full-dataset generation is selected.
* Display the completed CLIP video representation summary for verification.


In [ ]:
# ============================================================
# Step 9: Generate CLIP Video Representation Summary Report
# ============================================================

import shutil
import pandas as pd

print("Generating CLIP video representation summary report...")

if "clip_video_representation_df" not in globals():
    raise NameError(
        "clip_video_representation_df was not found. Run Step 6 first."
    )

if "clip_video_columns" not in globals():
    raise NameError(
        "clip_video_columns was not found. Run Step 6 first."
    )

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

input_splits = sorted(
    set(
        split
        for split_string in clip_video_representation_df["split"].astype(str)
        for split in split_string.split(",")
    )
)

missing_embedding_values = (
    clip_video_representation_df[clip_video_columns]
    .isna()
    .sum()
    .sum()
)

summary_rows = [
    {"metric": "artifact_scope", "value": "shared"},
    {"metric": "dataset_mode", "value": dataset_mode_label},
    {"metric": "input_splits", "value": ", ".join(input_splits)},
    {"metric": "representation_type", "value": "clip_video_representation"},
    {"metric": "clip_video_model", "value": CLIP_VIDEO_MODEL_NAME},
    {"metric": "clip_video_frames_per_video", "value": CLIP_VIDEO_FRAMES_PER_VIDEO},
    {"metric": "clip_video_frame_batch_size", "value": CLIP_VIDEO_FRAME_BATCH_SIZE},
    {"metric": "clip_video_pooling_method", "value": CLIP_VIDEO_POOLING_METHOD},
    {"metric": "representation_scope", "value": CLIP_VIDEO_REPRESENTATION_SCOPE},
    {"metric": "representation_source", "value": CLIP_VIDEO_REPRESENTATION_SOURCE},
    {"metric": "representation_records", "value": len(clip_video_representation_df)},
    {"metric": "unique_videos", "value": clip_video_representation_df["video"].nunique()},
    {"metric": "embedding_dimensions", "value": len(clip_video_columns)},
    {"metric": "missing_embedding_values", "value": int(missing_embedding_values)},
    {
        "metric": "minimum_frames_sampled",
        "value": int(clip_video_representation_df["frames_sampled"].min()),
    },
    {
        "metric": "maximum_frames_sampled",
        "value": int(clip_video_representation_df["frames_sampled"].max()),
    },
]

clip_video_summary_df = pd.DataFrame(summary_rows)

CLIP_VIDEO_LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

clip_video_summary_df.to_csv(
    CLIP_VIDEO_SUMMARY_LOCAL_CSV,
    index=False,
)

if not CLIP_VIDEO_SUMMARY_LOCAL_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create local summary file: "
        f"{CLIP_VIDEO_SUMMARY_LOCAL_CSV}"
    )

summary_drive_artifact_written = False

if GENERATE_FULL_DATASET:

    CLIP_VIDEO_REPRESENTATIONS_DRIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CLIP_VIDEO_SUMMARY_LOCAL_CSV,
        CLIP_VIDEO_SUMMARY_DRIVE_CSV,
    )

    if not CLIP_VIDEO_SUMMARY_DRIVE_CSV.exists():
        raise FileNotFoundError(
            f"Failed to create Drive summary file: "
            f"{CLIP_VIDEO_SUMMARY_DRIVE_CSV}"
        )

    summary_drive_artifact_written = True

else:

    print(
        "Development mode complete. Local summary report was "
        "created, but the persistent shared Drive summary was not "
        "overwritten."
    )

print("CLIP video representation summary report saved.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Local summary file : {CLIP_VIDEO_SUMMARY_LOCAL_CSV}")

if summary_drive_artifact_written:
    print(f"Drive summary file : {CLIP_VIDEO_SUMMARY_DRIVE_CSV}")
else:
    print("Drive summary file : not written in development mode")

display(clip_video_summary_df)


### 🔷 Step 10 — Display CLIP Video Representations

* Randomly select representative CLIP video representation records for inspection.
* Display dataset execution mode and input dataset splits.
* Display representation metadata and a subset of embedding dimensions.
* Verify that one representation was generated for each selected video.
* Confirm that the generated representation dataset is suitable for downstream representation-based VideoQA experiments.


In [ ]:
# ============================================================
# Step 10: Display CLIP Video Representations
# ============================================================

import pandas as pd

print("Displaying sample CLIP video representation records...")

if "clip_video_representation_df" not in globals():
    raise NameError(
        "clip_video_representation_df was not found. Run Step 6 first."
    )

if "clip_video_columns" not in globals():
    raise NameError(
        "clip_video_columns was not found. Run Step 6 first."
    )

sample_count = min(
    10,
    len(clip_video_representation_df),
)

sample_video_representation_df = (
    clip_video_representation_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

display_columns = [
    "record_id",
    "video",
    "split",
    "video_path",
    "representation_source",
    "frames_requested",
    "frames_sampled",
    "video_frame_count",
    "video_fps",
    "video_duration_seconds",
    "clip_video_model",
    "embedding_dimension",
    *clip_video_columns[:5],
]

print(f"Displaying {sample_count} CLIP video representation records...")

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)
print(f"Dataset mode          : {dataset_mode_label}")

input_splits = sorted(
    set(
        split
        for split_string in clip_video_representation_df["split"].astype(str)
        for split in split_string.split(",")
    )
)

print(f"Input splits          : {', '.join(input_splits)}")
print(f"CLIP model            : {CLIP_VIDEO_MODEL_NAME}")
print(f"Representation source : {CLIP_VIDEO_REPRESENTATION_SOURCE}")
print(f"Embedding dimensions  : {len(clip_video_columns)}")
print("Showing 10 sample records with the first 5 embedding columns.")

display(
    sample_video_representation_df[
        display_columns
    ]
)

print("\nCLIP Video Representation Summary")
print("-" * 60)
print(f"Representation records   : {len(clip_video_representation_df):,}")
print(f"Unique videos            : {clip_video_representation_df['video'].nunique():,}")
print(f"Embedding dimensions     : {len(clip_video_columns):,}")
print(f"Frames/video target      : {CLIP_VIDEO_FRAMES_PER_VIDEO}")
print(f"Minimum frames sampled   : {clip_video_representation_df['frames_sampled'].min():,}")
print(f"Maximum frames sampled   : {clip_video_representation_df['frames_sampled'].max():,}")


### 🔷 Step 11 — Notebook Summary

* Summarize the completed CLIP video representation generation process.
* Report the active shared representation configuration, dataset mode, CLIP video model, input splits, and frame sampling strategy.
* Summarize the generated video representation records.
* Report the number of unique videos and generated embedding dimensions.
* List the local and shared CLIP video representation artifacts generated by the notebook.
* Confirm that the generated CLIP video representations are ready for downstream representation-based VideoQA experiments.


In [ ]:
# ============================================================
# Step 11: Notebook Summary
# ============================================================

print("Notebook 06 complete.")
print("=" * 60)

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

input_splits = sorted(
    set(
        split
        for split_string in clip_video_representation_df["split"].astype(str)
        for split in split_string.split(",")
    )
)

print("\nCLIP Video Representations — Shared Configuration")
print("-" * 60)
print("Artifact scope           : shared")
print(f"Dataset mode             : {dataset_mode_label}")
print(f"CLIP video model         : {CLIP_VIDEO_MODEL_NAME}")
print(f"Input splits             : {', '.join(input_splits)}")

if GENERATE_FULL_DATASET:
    print("Development subset size  : not applicable")
else:
    print(f"Development subset size  : {development_video_subset_size}")

print(f"Frames per video         : {CLIP_VIDEO_FRAMES_PER_VIDEO}")
print(f"Frame pooling method     : {CLIP_VIDEO_POOLING_METHOD}")

print("\nRepresentation Dataset Summary")
print("-" * 60)
print(f"Representation records   : {len(clip_video_representation_df):,}")
print(f"Unique videos            : {clip_video_representation_df['video'].nunique():,}")
print(f"Embedding dimensions     : {len(clip_video_columns):,}")
print(f"Minimum frames sampled   : {clip_video_representation_df['frames_sampled'].min():,}")
print(f"Maximum frames sampled   : {clip_video_representation_df['frames_sampled'].max():,}")

print("\nShared Representation Outputs")
print("-" * 60)
print(f"Local video artifact     : {CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV}")
print(f"Local summary artifact   : {CLIP_VIDEO_SUMMARY_LOCAL_CSV}")

if GENERATE_FULL_DATASET:
    print(f"Shared video artifact    : {CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV}")
    print(f"Shared summary artifact  : {CLIP_VIDEO_SUMMARY_DRIVE_CSV}")
else:
    print("Shared video artifact    : not written in development mode")
    print("Shared summary artifact  : not written in development mode")

print(f"Shared Drive directory  : {CLIP_VIDEO_REPRESENTATIONS_DRIVE_DIR}")

print("\nNotebook 06 generated:")
print("- CLIP video representation dataset")
print("- CLIP video representation summary")
print("- Validated representation dataset")
print("- Sample representation records")

print("\nNotebook 06 outputs are ready for")
print("representation-based VideoQA experiments.")
